# Python Engineering Patterns – Interactive Demo

This notebook provides hands-on demonstrations of:
- Benchmark comparisons (iterrows vs vectorised, threading vs asyncio)
- Memory profiling examples
- Pattern usage demonstrations (collections, functools, design patterns)

**Prerequisites**: run `pip install -e ".[dev]"` from the project root and create `logs/`.

In [ ]:
import sys
import pathlib
import yaml

# Point Python at the project source modules
NOTEBOOK_DIR = pathlib.Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parents[2]  # python_engg_tmplates/
for src_dir in [
    PROJECT_ROOT / 'src' / 'collections_itertools',
    PROJECT_ROOT / 'src' / 'logging_patterns',
    PROJECT_ROOT / 'src' / 'concurrency',
    PROJECT_ROOT / 'src' / 'design_patterns',
    PROJECT_ROOT / 'src' / 'production_template',
]:
    sys.path.insert(0, str(src_dir))

# Load config
CONFIG_PATH = PROJECT_ROOT / 'config.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Ensure logs/ dir exists
(PROJECT_ROOT / 'logs').mkdir(exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Config loaded:', list(cfg.keys()))

---
## 1. Benchmark: `iterrows` vs Vectorised Pandas

In [ ]:
import time
import numpy as np
import pandas as pd

N = 100_000
np.random.seed(42)
df = pd.DataFrame({
    'a': np.random.randn(N),
    'b': np.random.randn(N),
    'c': np.random.randint(1, 10, N),
})

# -- iterrows (slow) --
t0 = time.perf_counter()
results_iterrows = []
for _, row in df.iterrows():
    results_iterrows.append(row['a'] * row['b'] + row['c'])
t_iterrows = time.perf_counter() - t0

# -- vectorised (fast) --
t0 = time.perf_counter()
results_vec = (df['a'] * df['b'] + df['c']).tolist()
t_vec = time.perf_counter() - t0

# -- apply (middle ground) --
t0 = time.perf_counter()
results_apply = df.apply(lambda r: r['a'] * r['b'] + r['c'], axis=1).tolist()
t_apply = time.perf_counter() - t0

assert results_iterrows[:5] == results_apply[:5], 'Results mismatch'

print(f"iterrows : {t_iterrows:.3f} s")
print(f"apply    : {t_apply:.3f} s  ({t_iterrows/t_apply:.1f}x faster than iterrows)")
print(f"vectorise: {t_vec:.4f} s  ({t_iterrows/t_vec:.0f}x faster than iterrows)")

In [ ]:
# Visualise the benchmark
import matplotlib.pyplot as plt

methods = ['iterrows', 'apply', 'vectorised']
times   = [t_iterrows, t_apply, t_vec]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(methods, times, color=['#e74c3c', '#f39c12', '#27ae60'])
ax.set_ylabel('Time (seconds)')
ax.set_title(f'Pandas row iteration vs vectorisation (N={N:,})')
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{t:.4f}s', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()
print('Key insight: vectorised operations use NumPy BLAS routines, bypassing Python interpreter overhead.')

---
## 2. Benchmark: Threading vs Asyncio for I/O-Bound Work

In [ ]:
import asyncio
import threading
import time
from concurrent.futures import ThreadPoolExecutor

NUM_TASKS = 50
TASK_LATENCY = 0.05  # 50 ms per task (simulated I/O)

# ---- Sequential ----
def sequential_io(n):
    for _ in range(n):
        time.sleep(TASK_LATENCY)

t0 = time.perf_counter()
sequential_io(NUM_TASKS)
t_seq = time.perf_counter() - t0
print(f'Sequential  : {t_seq:.3f} s')

# ---- ThreadPoolExecutor ----
def task(_):
    time.sleep(TASK_LATENCY)
    return True

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=cfg['concurrency']['thread_pool_size']) as pool:
    list(pool.map(task, range(NUM_TASKS)))
t_threads = time.perf_counter() - t0
print(f'ThreadPool  : {t_threads:.3f} s  (speedup {t_seq/t_threads:.1f}x)')

# ---- asyncio ----
async def async_task(_):
    await asyncio.sleep(TASK_LATENCY)
    return True

async def run_async():
    return await asyncio.gather(*[async_task(i) for i in range(NUM_TASKS)])

t0 = time.perf_counter()
asyncio.run(run_async())
t_async = time.perf_counter() - t0
print(f'asyncio     : {t_async:.3f} s  (speedup {t_seq/t_async:.1f}x)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
methods2 = ['Sequential', 'ThreadPool', 'asyncio']
times2   = [t_seq, t_threads, t_async]
bars2 = ax.bar(methods2, times2, color=['#e74c3c', '#3498db', '#9b59b6'])
ax.set_ylabel('Time (seconds)')
ax.set_title(f'I/O-bound concurrency comparison (N={NUM_TASKS} tasks × {TASK_LATENCY*1000:.0f} ms)')
for bar, t in zip(bars2, times2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{t:.3f}s', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()
print('Both threading and asyncio achieve near-ideal speedup for pure I/O-bound work.')
print('asyncio has lower per-task overhead (no OS thread creation) and better scaling to 1000s of tasks.')

---
## 3. Memory Profiling – Generator vs List

In [ ]:
import tracemalloc
import sys

N_ITEMS = 1_000_000

# ---- List approach ----
tracemalloc.start()
data_list = [i * i for i in range(N_ITEMS)]
peak_list = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
del data_list

# ---- Generator approach ----
tracemalloc.start()
gen = (i * i for i in range(N_ITEMS))
# consume without materialising
total = sum(gen)
peak_gen = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print(f'List comprehension peak memory : {peak_list / 1024**2:.2f} MB')
print(f'Generator expression peak memory: {peak_gen / 1024:.2f} KB')
print(f'Memory ratio (list/gen)         : {peak_list / peak_gen:.0f}x')

In [ ]:
# Memory profile over time simulation
import collections
import random

def measure_peak(fn):
    tracemalloc.start()
    fn()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1024  # KB

sizes = [1_000, 10_000, 100_000, 500_000, 1_000_000]
list_peaks = [measure_peak(lambda n=n: [i*i for i in range(n)]) for n in sizes]
gen_peaks  = [measure_peak(lambda n=n: sum(i*i for i in range(n))) for n in sizes]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sizes, list_peaks, 'o-', label='list comprehension', color='#e74c3c')
ax.plot(sizes, gen_peaks,  's--', label='generator', color='#27ae60')
ax.set_xlabel('Number of items')
ax.set_ylabel('Peak memory (KB)')
ax.set_title('Memory usage: list vs generator')
ax.legend()
ax.set_xscale('log')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

---
## 4. Collections Patterns Demo

In [ ]:
from collections import Counter, defaultdict, deque, namedtuple, ChainMap

# --- Counter: word frequency ---
text = 'to be or not to be that is the question whether tis nobler in the mind'
word_counts = Counter(text.split())
print('Top 5 words:', word_counts.most_common(5))

# --- defaultdict: inverted index ---
docs = {'doc1': 'python engineering patterns', 'doc2': 'python async patterns', 'doc3': 'design patterns'}
index = defaultdict(list)
for doc, content in docs.items():
    for word in content.split():
        index[word].append(doc)
print('Inverted index for "patterns":', index['patterns'])
print('Inverted index for "python"   :', index['python'])

# --- deque: sliding window ---
window_size = cfg['collections']['sliding_window_size']
data = [1, 3, 2, 5, 4, 6, 8, 7, 9, 10]
window = deque(maxlen=window_size)
avgs = []
for v in data:
    window.append(v)
    if len(window) == window_size:
        avgs.append(round(sum(window)/window_size, 2))
print(f'Sliding avg (window={window_size}):', avgs)

# --- namedtuple ---
Experiment = namedtuple('Experiment', ['name', 'accuracy', 'loss'])
exp = Experiment('bert-base', 0.923, 0.0341)
print(f'Experiment: {exp.name}  acc={exp.accuracy:.3f}  loss={exp.loss:.4f}')

# --- ChainMap: config layering ---
defaults = {'debug': False, 'workers': 2, 'timeout': 30}
overrides = {'workers': 8, 'debug': True}
merged = ChainMap(overrides, defaults)
print('Config workers:', merged['workers'], '  debug:', merged['debug'], '  timeout:', merged['timeout'])

---
## 5. functools Patterns Demo

In [ ]:
import functools
import time

# --- lru_cache: memoised Fibonacci ---
@functools.lru_cache(maxsize=cfg['collections']['lru_cache_size'])
def fib(n):
    if n < 2: return n
    return fib(n-1) + fib(n-2)

t0 = time.perf_counter()
result = fib(40)
cold = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
fib(40)
warm = (time.perf_counter() - t0) * 1000

print(f'fib(40) = {result}')
print(f'Cold call : {cold:.3f} ms')
print(f'Warm call : {warm:.5f} ms  (speedup {cold/warm:.0f}x)')
print(f'Cache info: {fib.cache_info()}')

In [ ]:
# --- cached_property ---
import math, time

class Stats:
    def __init__(self, data):
        self._data = data
    
    @functools.cached_property
    def summary(self):
        time.sleep(0.05)  # simulate expensive computation
        n = len(self._data)
        mean = sum(self._data) / n
        std = math.sqrt(sum((x-mean)**2 for x in self._data) / n)
        return {'n': n, 'mean': round(mean, 4), 'std': round(std, 4)}

s = Stats(list(range(1, 1001)))

t0 = time.perf_counter(); _ = s.summary; t1 = time.perf_counter()
print(f'First access  : {(t1-t0)*1000:.1f} ms  result={s.summary}')

t0 = time.perf_counter(); _ = s.summary; t1 = time.perf_counter()
print(f'Second access : {(t1-t0)*1000:.5f} ms  (cached)')

In [ ]:
# --- singledispatch ---
@functools.singledispatch
def process(obj):
    raise NotImplementedError(f'No handler for {type(obj).__name__}')

@process.register(int)
def _(obj): return f'INT: {obj * 2}'

@process.register(str)
def _(obj): return f'STR: {obj.upper()}'

@process.register(list)
def _(obj): return f'LIST: len={len(obj)} sum={sum(obj) if obj and isinstance(obj[0], (int,float)) else "n/a"}'

for val in [42, 'hello', [1, 2, 3, 4, 5]]:
    print(f'  process({val!r}) -> {process(val)}')

---
## 6. Design Patterns Demo

In [ ]:
# --- Singleton demo ---
import threading

class SingletonMeta(type):
    _instances = {}
    _lock = threading.Lock()
    
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            with cls._lock:
                if cls not in cls._instances:
                    cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class AppConfig(metaclass=SingletonMeta):
    def __init__(self, data=None):
        if not hasattr(self, '_init'):
            self._data = data or {}
            self._init = True
    
    def get(self, key, default=None):
        return self._data.get(key, default)

c1 = AppConfig({'env': 'prod', 'workers': 8})
c2 = AppConfig({'ignored': True})
print(f'c1 is c2: {c1 is c2}  (singleton)')
print(f'c2.get("env"): {c2.get("env")}  (from c1 init)')

In [ ]:
# --- QueryBuilder (fluent interface) ---
class QueryBuilder:
    def __init__(self):
        self._table = ''; self._cols = []; self._where = []; self._limit = None
    
    def from_table(self, t):
        self._table = t; return self
    def select(self, *c):
        self._cols.extend(c); return self
    def where(self, cond):
        self._where.append(cond); return self
    def limit(self, n):
        self._limit = n; return self
    def build(self):
        cols = ', '.join(self._cols) if self._cols else '*'
        sql = f'SELECT {cols} FROM {self._table}'
        if self._where: sql += ' WHERE ' + ' AND '.join(self._where)
        if self._limit: sql += f' LIMIT {self._limit}'
        return sql

query = (
    QueryBuilder()
    .from_table('users')
    .select('id', 'name', 'email')
    .where('active = TRUE')
    .where('role = "admin"')
    .limit(25)
    .build()
)
print('Built query:', query)

In [ ]:
# --- Retry decorator with exponential backoff ---
import functools, random, time, logging

def retry(max_attempts, backoff_factor, jitter=True, exceptions=(Exception,)):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts: raise
                    wait = backoff_factor * (2 ** (attempt - 1))
                    if jitter: wait += random.uniform(0, wait * 0.5)
                    print(f'  Attempt {attempt}/{max_attempts} failed: {e}; retry in {wait:.3f}s')
                    time.sleep(wait)
        return wrapper
    return decorator

call_n = [0]

@retry(
    max_attempts=cfg['retry']['max_attempts'],
    backoff_factor=0.05,  # short for demo
    jitter=cfg['retry']['jitter'],
    exceptions=(ValueError,)
)
def unstable_api():
    call_n[0] += 1
    if call_n[0] < 3:
        raise ValueError(f'Transient error on call #{call_n[0]}')
    return f'Success on call #{call_n[0]}'

result = unstable_api()
print('Result:', result)

---
## 7. Production Metrics Demo

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'production_template'))
from metrics import MetricsRegistry
import random, math

metrics_cfg = cfg['metrics']
registry = MetricsRegistry(
    maxlen=metrics_cfg['histogram_maxlen'],
    percentiles=metrics_cfg['percentiles']
)

req_counter  = registry.counter('demo.requests', 'Total requests')
err_counter  = registry.counter('demo.errors', 'Total errors')
active_gauge = registry.gauge('demo.active_connections', 'Active connections')
lat_hist     = registry.histogram('demo.latency_ms', 'Request latency ms')

rng = random.Random(42)
for i in range(500):
    req_counter.increment()
    active_gauge.set(rng.randint(10, 100))
    lat = rng.lognormvariate(3.5, 0.8)  # log-normal latency
    lat_hist.observe(lat)
    if rng.random() < 0.04:
        err_counter.increment()

# Print report
for snap in registry.report():
    if snap['type'] == 'counter':
        print(f"COUNTER  {snap['name']:<40}  value={snap['value']}")
    elif snap['type'] == 'gauge':
        print(f"GAUGE    {snap['name']:<40}  value={snap['value']:.1f}")
    elif snap['type'] == 'histogram':
        p = snap['percentiles']
        print(f"HIST     {snap['name']:<40}  count={snap['count']}  "
              f"mean={snap['mean']:.2f}ms  p50={p['p50']:.2f}  p95={p['p95']:.2f}  p99={p['p99']:.2f}")

In [ ]:
# Visualise latency distribution
snap = registry.histogram('demo.latency_ms').snapshot()

# Re-generate the raw observations for plotting
rng2 = random.Random(42)
latencies = [rng2.lognormvariate(3.5, 0.8) for _ in range(500)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(latencies, bins=50, color='#3498db', edgecolor='white', alpha=0.8)
pcts = snap['percentiles']
for label, val, color in [('p50', pcts['p50'], 'green'), ('p95', pcts['p95'], 'orange'), ('p99', pcts['p99'], 'red')]:
    axes[0].axvline(val, color=color, linestyle='--', label=f'{label}={val:.0f}ms')
axes[0].set_xlabel('Latency (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('Request Latency Distribution')
axes[0].legend()

# Percentile bar chart
pct_labels = list(pcts.keys())
pct_values = list(pcts.values())
axes[1].barh(pct_labels, pct_values, color='#9b59b6')
axes[1].set_xlabel('Latency (ms)')
axes[1].set_title('Latency Percentiles')
for i, v in enumerate(pct_values):
    axes[1].text(v + 1, i, f'{v:.1f}ms', va='center')

plt.tight_layout()
plt.show()

---
## 8. TTL Cache Behaviour

In [ ]:
import time, threading
from collections import OrderedDict
from typing import Optional, Any

class TTLCache:
    def __init__(self, max_size, ttl):
        self._max = max_size; self._ttl = ttl
        self._store = OrderedDict()
        self._lock = threading.Lock()
        self._hits = self._misses = 0
    
    def get(self, key):
        with self._lock:
            if key not in self._store:
                self._misses += 1; return None
            val, exp = self._store[key]
            if time.monotonic() > exp:
                del self._store[key]; self._misses += 1; return None
            self._store.move_to_end(key); self._hits += 1; return val
    
    def put(self, key, value):
        with self._lock:
            if key in self._store: self._store.move_to_end(key)
            self._store[key] = (value, time.monotonic() + self._ttl)
            while len(self._store) > self._max:
                self._store.popitem(last=False)
    
    def hit_rate(self):
        total = self._hits + self._misses
        return self._hits / total if total else 0.0

cache = TTLCache(max_size=cfg['cache']['max_size'], ttl=cfg['cache']['ttl_seconds'])

# Simulate cache usage
import random
rng = random.Random(0)
for _ in range(1000):
    key = f'key-{rng.randint(0, 50)}'  # key space smaller than cache -> many hits
    if cache.get(key) is None:
        cache.put(key, rng.random())

print(f'Cache hit rate : {cache.hit_rate():.1%}')
print(f'Cache size     : {len(cache._store)}')

# Demonstrate TTL expiry
short_cache = TTLCache(max_size=10, ttl=0.05)
short_cache.put('temp', 42)
print(f'Before TTL: {short_cache.get("temp")}')
time.sleep(0.06)
print(f'After TTL : {short_cache.get("temp")}  (expired)')

---
## 9. itertools Patterns Demo

In [ ]:
import itertools, operator

# --- groupby ---
records = sorted([
    {'name': 'Alice', 'dept': 'eng'}, {'name': 'Bob', 'dept': 'eng'},
    {'name': 'Carol', 'dept': 'hr'}, {'name': 'Dave', 'dept': 'hr'},
    {'name': 'Eve', 'dept': 'finance'},
], key=lambda r: r['dept'])

print('groupby dept:')
for dept, members in itertools.groupby(records, key=lambda r: r['dept']):
    print(f'  {dept}: {[m["name"] for m in members]}')

# --- accumulate ---
data = [3, 1, 4, 1, 5, 9, 2, 6]
print('\nRunning max:', list(itertools.accumulate(data, max)))
print('Running sum:', list(itertools.accumulate(data)))

# --- product (hyperparameter grid) ---
lrs = [1e-3, 1e-4]
bs  = [32, 64]
opts = ['adam', 'sgd']
grid = list(itertools.product(lrs, bs, opts))
print(f'\nHyperparameter grid ({len(grid)} combos):')
for lr, b, opt in grid:
    print(f'  lr={lr}  batch={b}  opt={opt}')

In [ ]:
# --- batch generator ---
def batch_generator(iterable, n):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == n:
            yield batch; batch = []
    if batch:
        yield batch

dataset = list(range(1, 26))
print('dataset:', dataset)
print('\nBatches (size=7):')
for i, b in enumerate(batch_generator(dataset, 7)):
    print(f'  batch {i+1}: {b}')

---
## Summary

| Pattern | Key Takeaway |
|---------|-------------|
| `iterrows` vs vectorised | Use NumPy/pandas vectorisation; avoid Python loops over DataFrame rows |
| Threading vs asyncio | Both work for I/O-bound; asyncio scales to thousands of tasks with lower overhead |
| Generator vs list | Generators have O(1) memory vs O(n) for lists; use for large pipelines |
| `lru_cache` | Near-zero cost for repeated calls; inspect with `cache_info()` |
| Singleton | Metaclass approach is thread-safe; use for process-wide shared state |
| Builder | Fluent interface for complex object construction; eliminates telescope constructors |
| Retry decorator | Exponential backoff + jitter reduces thundering herd on retry storms |
| TTL Cache | OrderedDict enables O(1) LRU eviction; TTL prevents stale data |
| Metrics | In-process counters/histograms cost nanoseconds; compute percentiles on demand |